In [16]:
#configuration
from pathlib import Path

ROOT_DIR = Path.cwd().parent.parent

DATASET_PATH = ROOT_DIR / "eval" / "retrieval_evaluation" / "data" / "retrieval_eval.json"
DOC_STORE_PATH = ROOT_DIR / "data" / "faiss_storage" / "doc_store.json"
CONTEXTS_PATH = ROOT_DIR / "database" / "contexts.json"

RESULTS_DIR = ROOT_DIR / "retrieval_evaluation" / "results"

print("ROOT_DIR:", ROOT_DIR)
print("DATASET:", DATASET_PATH, DATASET_PATH.exists())
print("DOC_STORE:", DOC_STORE_PATH, DOC_STORE_PATH.exists())
print("CONTEXTS:", CONTEXTS_PATH, CONTEXTS_PATH.exists())
print("RESULTS:", RESULTS_DIR)
K_VALUES = [1, 3, 5, 10, 20]

ROOT_DIR: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit
DATASET: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/data/retrieval_eval.json True
DOC_STORE: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/data/faiss_storage/doc_store.json True
CONTEXTS: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/database/contexts.json True
RESULTS: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/retrieval_evaluation/results


In [27]:
#load dataset
import json

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Số lượng queries:", len(eval_data))
print("Query đầu tiên:")
print(eval_data[0])

Số lượng queries: 182
Query đầu tiên:
{'id': 'd55bbab59cdd93ee833f82d41b03669486dfc8f592e931dd572dcdcd0edbbb59', 'query': 'Mùng 1 âm lịch có tên gọi là gì?', 'relevant_documents': [{'document_id': '8cd35d91-d78c-4883-b8d6-ceea61bc3a14', 'relevance': 2}]}


In [23]:
#canonical relevance
def get_relevance_map(record):
    return {
        item["document_id"]: item["relevance"]
        for item in record["relevant_documents"]
    }

In [38]:
print(RETRIEVER_PATH.read_text(encoding="utf-8"))

/home/sotatek/Projects/Tools/Notebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Retriever imported successfully!


In [48]:
#mapping FAISS index -> canonical document_id

faiss_to_document_id = {}

for i in range(len(contexts)):
    doc = doc_store["docs"][str(i)]
    document_id = doc["metadata"]["document_id"]
    
    faiss_to_document_id[i] = document_id

print("Number of mappings:", len(faiss_to_document_id))

for i in range(min(5, len(faiss_to_document_id))):
    print(f"FAISS {i} -> {faiss_to_document_id[i]}")

Number of mappings: 21
FAISS 0 -> 8cd35d91-d78c-4883-b8d6-ceea61bc3a14
FAISS 1 -> acecae37-a0a9-44d4-9eae-54d0fd4f78e5
FAISS 2 -> 847feffa-0973-4af4-ad32-ef1a5b0ecb82
FAISS 3 -> f651af0d-6291-4e54-ac9f-5a2016a6142f
FAISS 4 -> c862facb-f2b4-404b-887b-d5fda59a948c


In [49]:
#Tạo hàm metric
import math


def recall_at_k(retrieved_ids, relevant_ids, k):
    """
    Recall@K
    """
    relevant = set(relevant_ids)
    retrieved = set(retrieved_ids[:k])
    
    if not relevant:
        return 0.0
    
    return len(retrieved & relevant) / len(relevant)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """
    HitRate@K
    """
    relevant = set(relevant_ids)
    retrieved = set(retrieved_ids[:k])
    
    return 1.0 if retrieved & relevant else 0.0


def reciprocal_rank(retrieved_ids, relevant_ids):
    """
    Reciprocal Rank của relevant document đầu tiên.
    """
    relevant = set(relevant_ids)
    
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    
    return 0.0


def ndcg_at_k(retrieved_ids, relevance_map, k):
    """
    nDCG@K với graded relevance.
    """
    
    # DCG
    dcg = 0.0
    
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        relevance = relevance_map.get(doc_id, 0)
        
        dcg += (
            (2 ** relevance - 1)
            / math.log2(rank + 1)
        )
    
    # Ideal DCG
    ideal_relevances = sorted(
        relevance_map.values(),
        reverse=True
    )[:k]
    
    idcg = 0.0
    
    for rank, relevance in enumerate(
        ideal_relevances,
        start=1
    ):
        idcg += (
            (2 ** relevance - 1)
            / math.log2(rank + 1)
        )
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


print("Metric functions loaded successfully!")

Metric functions loaded successfully!


In [51]:
#test metric
record = eval_data[0]

relevant_ids = [
    item["document_id"]
    for item in record["relevant_documents"]
]

relevance_map = {
    item["document_id"]: item["relevance"]
    for item in record["relevant_documents"]
}

result = retriever.retrieve(
    record["query"],
    top_k=50
)

retrieved_ids = []

for doc in result["documents"]:
    faiss_id = int(doc["id"].split("_")[1])
    
    canonical_id = faiss_to_document_id[faiss_id]
    
    retrieved_ids.append(canonical_id)


print("=== METRIC TEST ===")

print("Query:")
print(record["query"])

print("\nRelevant:")
print(relevant_ids)

print("\nRetrieved:")
for rank, doc_id in enumerate(retrieved_ids, start=1):
    print(rank, doc_id)

print("\nMetrics:")

for k in [1, 3, 5, 10, 20, 50]:
    print(
        f"Recall@{k}:",
        round(recall_at_k(retrieved_ids, relevant_ids, k), 4)
    )

print(
    "\nMRR:",
    round(reciprocal_rank(retrieved_ids, relevant_ids), 4)
)

for k in [1, 3, 5, 10, 20, 50]:
    print(
        f"nDCG@{k}:",
        round(
            ndcg_at_k(
                retrieved_ids,
                relevance_map,
                k
            ),
            4
        )
    )

=== METRIC TEST ===
Query:
Mùng 1 âm lịch có tên gọi là gì?

Relevant:
['8cd35d91-d78c-4883-b8d6-ceea61bc3a14']

Retrieved:
1 8cd35d91-d78c-4883-b8d6-ceea61bc3a14
2 4fc644e2-3ceb-46f9-bdaf-f0c2f601737d
3 c862facb-f2b4-404b-887b-d5fda59a948c
4 d493a46e-0681-4223-924e-75d4dd9940b0
5 acecae37-a0a9-44d4-9eae-54d0fd4f78e5
6 4d358a44-a0d3-4fa5-9a80-7e86b0b95fe6
7 103c7e98-bbed-433b-a036-892704308e20
8 f99d95c8-92c4-4295-ae66-ea252db809c4
9 1429147f-12b8-4bf5-abe8-823e8a1547b0
10 bc46e4cb-56b2-4531-955a-294b1db90c93
11 3d9d0d94-3438-43a8-83b0-79814b8b0f31
12 3c0b2722-5ec6-429d-b169-5f5e7d2546ad
13 f651af0d-6291-4e54-ac9f-5a2016a6142f
14 847feffa-0973-4af4-ad32-ef1a5b0ecb82
15 eb9d19d8-7847-480f-9c2e-6949a5b00101
16 08d3d824-f2d6-4d51-919f-fab1eaa1c916
17 3f3787bd-d19c-4d60-860e-a28f36958a79
18 4eede11b-8b57-43e0-8e01-0ad1be94aa2a
19 ea00603f-ff3f-4bc8-acbe-718d4a0576e9
20 1db81443-d32c-4671-b64a-d17aea33dd6e
21 28ff1e30-5f5b-47ce-8e37-dfbed9f95efe

Metrics:
Recall@1: 1.0
Recall@3: 1.0
Recall@

In [53]:
#Retrieval evaluation
import time
import pandas as pd


K_VALUES = [1, 3, 5, 10, 20]

per_query_results = []

print("=== START RETRIEVAL EVALUATION ===")
print("Total queries:", len(eval_data))
print("Top-K retrieval:", 20)

for query_idx, record in enumerate(eval_data, start=1):
    
    query_id = record["id"]
    query = record["query"]
    
    # Gold documents
    relevant_ids = [
        item["document_id"]
        for item in record["relevant_documents"]
    ]
    
    relevance_map = {
        item["document_id"]: item["relevance"]
        for item in record["relevant_documents"]
    }
    
    # Retrieve once
    start_time = time.perf_counter()
    
    result = retriever.retrieve(
        query,
        top_k=20
    )
    
    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000
    
    # Convert FAISS IDs -> canonical document IDs
    retrieved_ids = []
    
    for doc in result["documents"]:
        
        faiss_id = int(
            doc["id"].split("_")[1]
        )
        
        canonical_id = faiss_to_document_id[
            faiss_id
        ]
        
        retrieved_ids.append(canonical_id)
    
    # Basic information
    row = {
        "id": query_id,
        "query": query,
        "latency_ms": latency_ms,
        "effective_k": len(retrieved_ids),
    }
    
    # Metrics at each K
    for k in K_VALUES:
        
        row[f"recall@{k}"] = recall_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )
        
        row[f"hit_rate@{k}"] = hit_rate_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )
        
        row[f"ndcg@{k}"] = ndcg_at_k(
            retrieved_ids,
            relevance_map,
            k
        )
    
    row["mrr"] = reciprocal_rank(
        retrieved_ids,
        relevant_ids
    )
    
    # Store ranking for debugging
    row["retrieved_documents"] = retrieved_ids
    row["relevant_documents"] = relevant_ids
    
    per_query_results.append(row)
    
    # Progress
    if query_idx % 10 == 0 or query_idx == 1:
        print(
            f"Processed {query_idx}/{len(eval_data)} "
            f"queries"
        )

print("\n=== EVALUATION FINISHED ===")
print("Queries evaluated:", len(per_query_results))

=== START RETRIEVAL EVALUATION ===
Total queries: 182
Top-K retrieval: 20
Processed 1/182 queries
Processed 10/182 queries
Processed 20/182 queries
Processed 30/182 queries
Processed 40/182 queries
Processed 50/182 queries
Processed 60/182 queries
Processed 70/182 queries
Processed 80/182 queries
Processed 90/182 queries
Processed 100/182 queries
Processed 110/182 queries
Processed 120/182 queries
Processed 130/182 queries
Processed 140/182 queries
Processed 150/182 queries
Processed 160/182 queries
Processed 170/182 queries
Processed 180/182 queries

=== EVALUATION FINISHED ===
Queries evaluated: 182


In [54]:
#chuyển thành Dataframe
results_df = pd.DataFrame(per_query_results)

print("Shape:", results_df.shape)

print("\nColumns:")
print(results_df.columns.tolist())

print("\nFirst 5 rows:")
display(results_df.head())

Shape: (182, 22)

Columns:
['id', 'query', 'latency_ms', 'effective_k', 'recall@1', 'hit_rate@1', 'ndcg@1', 'recall@3', 'hit_rate@3', 'ndcg@3', 'recall@5', 'hit_rate@5', 'ndcg@5', 'recall@10', 'hit_rate@10', 'ndcg@10', 'recall@20', 'hit_rate@20', 'ndcg@20', 'mrr', 'retrieved_documents', 'relevant_documents']

First 5 rows:


,id,query,latency_ms,effective_k,recall@1,hit_rate@1,ndcg@1,recall@3,hit_rate@3,ndcg@3,...,ndcg@5,recall@10,hit_rate@10,ndcg@10,recall@20,hit_rate@20,ndcg@20,mrr,retrieved_documents,relevant_documents
0,d55bbab59cdd93ee833f82d41b03669486dfc8f592e931...,Mùng 1 âm lịch có tên gọi là gì?,83.856096,20,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,"[8cd35d91-d78c-4883-b8d6-ceea61bc3a14, 4fc644e...",[8cd35d91-d78c-4883-b8d6-ceea61bc3a14]
1,86c1ef7117b244512d79fac15000508c792b354c586452...,Ngày Mùng 1 âm lịch liên quan đến điều kiện nà...,6.251541,20,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,"[8cd35d91-d78c-4883-b8d6-ceea61bc3a14, f99d95c...",[8cd35d91-d78c-4883-b8d6-ceea61bc3a14]
2,354350bd3418e59612e89a3ac74f8595f67ec2825b46bf...,Tại sao gọi ngày Mùng 1 âm lịch là Sóc?,4.945538,20,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,"[8cd35d91-d78c-4883-b8d6-ceea61bc3a14, c862fac...",[8cd35d91-d78c-4883-b8d6-ceea61bc3a14]
3,e63979be0e7fa83b84a09b63ef15379c979eadf2db684e...,"Trong một chu kỳ trăng, ngày nào sẽ không có t...",2.793864,20,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.356207,1.0,1.0,0.356207,0.166667,"[f651af0d-6291-4e54-ac9f-5a2016a6142f, 4eede11...",[8cd35d91-d78c-4883-b8d6-ceea61bc3a14]
4,89bab97aa0f4fd5efd60cae830559a87eea70abf90c46a...,Tại sao ngày Mùng 1 âm lịch thường không có tr...,2.461161,20,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.000000,1.0,1.0,1.000000,1.000000,"[8cd35d91-d78c-4883-b8d6-ceea61bc3a14, 4d358a4...",[8cd35d91-d78c-4883-b8d6-ceea61bc3a14]


In [55]:
summary_rows = []

for k in K_VALUES:
    
    summary_rows.append({
        "k": k,
        "recall": results_df[f"recall@{k}"].mean(),
        "hit_rate": results_df[f"hit_rate@{k}"].mean(),
        "ndcg": results_df[f"ndcg@{k}"].mean(),
    })


summary_df = pd.DataFrame(summary_rows)

print("=== RETRIEVAL SUMMARY ===")

display(summary_df)

=== RETRIEVAL SUMMARY ===


,k,recall,hit_rate,ndcg
0,1,0.368132,0.368132,0.368132
1,3,0.554945,0.554945,0.475926
2,5,0.681319,0.681319,0.528426
3,10,0.890110,0.890110,0.597314
4,20,0.994505,0.994505,0.623959


In [57]:
#Thống kê latency
latency_summary = {
    "num_queries": len(results_df),
    "avg_latency_ms": results_df["latency_ms"].mean(),
    "median_latency_ms": results_df["latency_ms"].median(),
    "p95_latency_ms": results_df["latency_ms"].quantile(0.95),
    "min_latency_ms": results_df["latency_ms"].min(),
    "max_latency_ms": results_df["latency_ms"].max(),
}

latency_df = pd.DataFrame([latency_summary])

print("=== LATENCY SUMMARY ===")
display(latency_df)

=== LATENCY SUMMARY ===


,num_queries,avg_latency_ms,median_latency_ms,p95_latency_ms,min_latency_ms,max_latency_ms
0,182,6.895956,4.171896,11.829779,2.175048,83.856096


In [63]:
RESULTS_DIR = Path.cwd() / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Results directory:", RESULTS_DIR)
print("Exists:", RESULTS_DIR.exists())

Results directory: /home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results
Exists: True


In [64]:
import json


# 1. Per-query CSV
per_query_csv = RESULTS_DIR / "retrieval_per_query.csv"

# Không lưu list trực tiếp vào CSV
csv_df = results_df.copy()

csv_df["retrieved_documents"] = csv_df[
    "retrieved_documents"
].apply(lambda x: json.dumps(x, ensure_ascii=False))

csv_df["relevant_documents"] = csv_df[
    "relevant_documents"
].apply(lambda x: json.dumps(x, ensure_ascii=False))

csv_df.to_csv(
    per_query_csv,
    index=False,
    encoding="utf-8-sig"
)


# 2. Summary CSV
summary_csv = RESULTS_DIR / "retrieval_metrics.csv"

summary_df.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig"
)


# 3. Latency CSV
latency_csv = RESULTS_DIR / "retrieval_latency.csv"

latency_df.to_csv(
    latency_csv,
    index=False,
    encoding="utf-8-sig"
)


# 4. Full JSON result
results_json = RESULTS_DIR / "retrieval_per_query.json"

with open(
    results_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        per_query_results,
        f,
        ensure_ascii=False,
        indent=2
    )


# 5. Summary JSON
summary_json = RESULTS_DIR / "retrieval_metrics.json"

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "num_queries": len(results_df),
            "k_values": K_VALUES,
            "metrics": summary_rows,
            "latency": latency_summary
        },
        f,
        ensure_ascii=False,
        indent=2
    )


print("=== FILES SAVED ===")

for path in [
    per_query_csv,
    summary_csv,
    latency_csv,
    results_json,
    summary_json
]:
    print(path)

=== FILES SAVED ===
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results/retrieval_per_query.csv
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results/retrieval_metrics.csv
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results/retrieval_latency.csv
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results/retrieval_per_query.json
/home/sotatek/Projects/Tools/Notebook/Luong.Nguyen/NEU-AI/ai-espirit/eval/retrieval_evaluation/results/retrieval_metrics.json
